In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 12 — A FUNÇÃO LOGÍSTICA: REGRESSÃO PARA CLASSIFICAÇÃO

> "O infinito é um conceito difícil de trabalhar. A beleza da função logística é que ela pega o infinito e, gentilmente, o dobra para caber entre zero e um."
> — Anotação em um antigo livro de estatística

Minha jornada parecia me afastar do ponto de partida. Comecei na regressão, prevendo valores contínuos; agora, na classificação, explorara probabilidades, geometria, regras e vizinhanças. Mundos distintos. Mas eu sabia que havia uma ponte — um algoritmo que usava a mecânica da regressão para resolver a classificação: a **Regressão Logística**.

O nome era um paradoxo que me intrigava. Ela pega a saída de uma regressão linear (que vai de −∞ a +∞) e a passa por uma função especial — a **sigmoide** — que a "espreme" para um valor entre 0 e 1: uma probabilidade bem-comportada. Era a peça que faltava, a ponte entre os dois mundos.

## Regressão Logística: Da Linha à Probabilidade

Apesar do nome, é um algoritmo de **classificação**. Ela parte da soma ponderada das *features* (z = β₀ + β₁x₁ + … + βₙxₙ) e aplica a **função sigmoide**:

**σ(z) = 1 / (1 + e^(−z))**

Quando z é muito positivo, σ(z) → 1; muito negativo, → 0; em z = 0, σ(z) = 0,5. O modelo aprende os coeficientes β que empurram a probabilidade para perto de 1 nos casos positivos e de 0 nos negativos.

**Interpretabilidade.** Como é linear no cerne, os coeficientes têm significado: representam a variação no **log-odds** da classe para cada aumento de uma unidade na *feature* (após padronização). Exponenciando um coeficiente obtém-se o *odds ratio*. É essa transparência que torna a Regressão Logística a primeira escolha em setores regulados como saúde e finanças.

Como todo modelo linear, ela é **sensível à escala** — daí o `StandardScaler` no *pipeline*.

#### Código 12.1: Treinando a Regressão Logística


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

# liblinear: solver eficiente para datasets pequenos como o nosso.
pipe = make_pipeline(StandardScaler(),
                     LogisticRegression(solver="liblinear", random_state=42))
r = cross_validate(pipe, X_train, y_train, cv=5,
                   scoring={"acuracia": "accuracy", "recall_Maligno": recall_maligno})

print(f"Acuracia media:        {r['test_acuracia'].mean():.4f}")
print(f"Recall Maligno medio:  {r['test_recall_Maligno'].mean():.4f}")

Acuracia media:        0.9688
Recall Maligno medio:  0.9548


Um resultado notável: **96,88%** de acurácia e **recall maligno de 95,48%** — o segundo melhor recall maligno entre os modelos individuais, logo atrás do SVM-RBF (96,05%) e à frente de KNN (94,4%) e Naive Bayes (90,4%). Um modelo linear simples, com a **melhor interpretabilidade** do arsenal, praticamente empata com os mais complexos. Isso sugere que a fronteira de decisão do problema é, em grande medida, linear.

#### Código 12.2: O Que o Modelo Aprendeu — Os Coeficientes


In [ ]:
import pandas as pd

pipe.fit(X_train, y_train)   # treina no conjunto de treino completo
modelo = pipe.named_steps["logisticregression"]

# O sklearn modela P(classe 1 = Benigno). Logo, coeficiente NEGATIVO
# aumenta a chance de MALIGNO; POSITIVO, a de BENIGNO.
coefs = pd.Series(modelo.coef_[0], index=X_train.columns).sort_values()

print("Mais puxam para MALIGNO (coeficientes negativos):")
print(coefs.head(5).round(4).to_string())
print("\nMais puxam para BENIGNO (coeficientes positivos):")
print(coefs.tail(5).round(4).to_string())

Mais puxam para MALIGNO (coeficientes negativos):
worst texture          -1.3691
radius error           -1.1705
worst concave points   -1.0688
area error             -1.0436
worst area             -0.9763

Mais puxam para BENIGNO (coeficientes positivos):
symmetry error             0.2244
texture error              0.2501
fractal dimension error    0.4808
compactness error          0.6359
mean compactness           0.7109


> **📘 PARA SABER — Coeficiente não é o mesmo que "importância"**
>
> Note que `worst texture` tem o maior coeficiente negativo aqui, mas no Capítulo 7 quem liderava a importância (Informação Mútua / Random Forest) era `worst perimeter`/`worst radius`. Não há contradição: o coeficiente mede o efeito **parcial** de uma *feature* mantidas as outras fixas, num contexto de forte multicolinearidade. Quando `worst radius`, `worst perimeter` e `worst area` carregam a mesma informação, o modelo linear "distribui" o crédito entre elas de forma que pode não refletir a importância isolada. Por isso usamos **várias lentes** de interpretação — e, nos Capítulos 18 a 20, ferramentas dedicadas (Permutation Importance, SHAP, LIME).

Ainda assim, o sinal é coerente com o domínio: medidas de **tamanho** (`worst area`), de **irregularidade** (`worst concave points`) e de **variação** (`worst texture`, `radius error`, `area error`) empurram o diagnóstico para maligno. O modelo aprendeu, à sua maneira, a mesma biologia que a EDA sugeriu.

> **📌 CHECKLIST DO CAPÍTULO 12**
>
> ✅ Entende por que a Regressão Logística é classificação, apesar do nome.
>
> ✅ Compreende o papel da **sigmoide** e o conceito de **log-odds**.
>
> ✅ Executou o Código 12.1: **96,9%** de acurácia e **95,5%** de recall maligno.
>
> ✅ Inspecionou os coeficientes e entende por que magnitude de coeficiente ≠ importância global.
>
> **⚠️ CRÍTICO:** A Regressão Logística oferece rara combinação de alta performance e alta interpretabilidade — por isso é frequentemente a primeira escolha em domínios auditáveis.

A ponte entre regressão e classificação estava construída, e era sólida. A Regressão Logística me deu uma sensação de controle que nenhum outro modelo proporcionara: eu não tinha só uma previsão, tinha uma probabilidade *e* uma explicação.

Com isso, minha exploração do arsenal estava completa: cinco abordagens fundamentalmente diferentes, um ranking claro de competidores. Mas até aqui eu tratara cada etapa — pré-processamento, treino, avaliação — como atos separados. Em produção, isso é frágil. Eu precisava uni-los numa estrutura única, coesa e robusta. Precisava construir uma arquitetura para a confiabilidade: os *Pipelines*.

**PARTE IV — AUTOMAÇÃO E ROBUSTEZ**
